In [0]:
%sql
-- Tạo schema silver nếu chưa có
CREATE SCHEMA IF NOT EXISTS workspace.silver;

-- Kiểm tra xem schema đã được tạo chưa
SHOW SCHEMAS IN workspace;


In [0]:
import pyspark.sql.functions as F

df_raw = spark.table("workspace.bronze.bronze_products")

# BỆNH 1: SOI KIỂU DỮ LIỆU (SCHEMA)
print("=== 1. KIỂU DỮ LIỆU HIỆN TẠI CỦA CỘT GIÁ TIỀN ===")
df_raw.select("product_name", "unit_cost_usd", "unit_price_usd").printSchema()

# BỆNH 2: XEM TẬN MẮT GIÁ TRỊ THỰC TẾ
print("=== 2. SOI GIÁ TRỊ THẬT TRONG BẢNG (CÓ DẤU $ VÀ KHOẢNG TRẮNG) ===")
display(df_raw.select("product_name", "unit_cost_usd", "unit_price_usd").limit(5))


In [0]:
import pyspark.sql.functions as F

# 1. Đọc bảng Bronze thô lên
df_bronze_products = spark.table("workspace.bronze.bronze_products")

# 2. BẮT ĐẦU CHỮA BỆNH: Bóc tách dấu $ và ép kiểu sang Double chuẩn
df_silver_products = (df_bronze_products
    .withColumn("unit_price_usd", F.regexp_replace(F.col("unit_price_usd"), "[$, ]", "").cast("double"))
    .withColumn("unit_cost_usd", F.regexp_replace(F.col("unit_cost_usd"), "[$, ]", "").cast("double"))
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("brand", F.trim(F.col("brand")))
    .withColumn("color", F.coalesce(F.trim(F.col("color")), F.lit("Unknown")))
    .withColumn("_updated_at", F.current_timestamp())
    .drop("_ingested_at", "_source_file")
)

# 3. Ghi ra bảng sạch Silver
(df_silver_products.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.silver_products"))

print(" ĐÃ CHỮA BỆNH THÀNH CÔNG VÀ LƯU VÀO: workspace.silver.silver_products!")



In [0]:
# TÍNH LẠI TỔNG GIÁ BÁN TRÊN BẢNG SILVER SẠCH:
spark.table("workspace.silver.silver_products").select(F.sum("unit_price_usd")).show()


In [0]:
import pyspark.sql.functions as F

df_stores_bronze = spark.table("workspace.bronze.bronze_stores")

print("=== 1. KIỂU DỮ LIỆU HIỆN TẠI CỦA BẢNG STORES ===")
df_stores_bronze.printSchema()

print("=== 2. SOI DỮ LIỆU VÀ DIỆN TÍCH CỬA HÀNG ===")
display(df_stores_bronze.select("storekey", "country", "state", "square_meters", "open_date").limit(5))


In [0]:
# CHỮA BỆNH VÀ LÀM GIÀU DỮ LIỆU BẢNG STORES (D_SHOP_FLC)
df_silver_stores = (df_stores_bronze
    # 1. Đổi tên storekey thành store_key chuẩn snake_case
    .withColumnRenamed("storekey", "store_key")
    # 2. Khử trùng lặp theo khóa chính store_key
    .dropDuplicates(["store_key"])
    # 3. Chuẩn hóa ngày mở cửa sang DateType
    .withColumn("open_date", F.to_date(F.col("open_date")))
    # 4. Fill giá trị 0 nếu square_meters bị null
    .withColumn("square_meters", F.coalesce(F.col("square_meters"), F.lit(0)))
    # 5. Thêm cột phân loại quy mô cửa hàng (Business Logic)
    .withColumn("store_size_category", 
        F.when(F.col("square_meters") < 500, "Small")
         .when(F.col("square_meters") <= 1500, "Medium")
         .otherwise("Large")
    )
    # 6. Cập nhật tem Audit
    .withColumn("_updated_at", F.current_timestamp())
    .drop("_ingested_at", "_source_file")
)

# Ghi ra bảng Delta tầng Silver
(df_silver_stores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.silver_stores"))

print(" ĐÃ TẠO THÀNH CÔNG BẢNG SẠCH: workspace.silver.silver_stores (D_SHOP)!")

# Hiển thị bảng sạch đã có thêm cột phân loại Small / Medium / Large
display(spark.table("workspace.silver.silver_stores").select("store_key", "country", "square_meters", "open_date", "store_size_category").limit(10))


In [0]:
import pyspark.sql.functions as F

df_cust_bronze = spark.table("workspace.bronze.bronze_customers")

print("=== 1. KIỂU DỮ LIỆU HIỆN TẠI CỦA KHÁCH HÀNG ===")
df_cust_bronze.printSchema()

print("=== 2. SOI DỮ LIỆU NGÀY SINH VÀ TÊN KHÁCH HÀNG ===")
display(df_cust_bronze.select("customerkey", "name", "gender", "city", "country", "birthday").limit(5))


In [0]:
# CHUẨN HÓA VÀ LÀM GIÀU DỮ LIỆU KHÁCH HÀNG
df_silver_customers = (df_cust_bronze
    # 1. Đổi tên khóa chính
    .withColumnRenamed("customerkey", "customer_key")
    # 2. Khử trùng lặp khách hàng
    .dropDuplicates(["customer_key"])
    # 3. Viết hoa chữ cái đầu họ tên
    .withColumn("name", F.initcap(F.trim(F.col("name"))))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("country", F.trim(F.col("country")))
    # 4. Ép ngày sinh sang DateType
    .withColumn("birthday", F.to_date(F.col("birthday")))
    # 5. TÍNH TUỔI VÀ PHÂN NHÓM TUỔI (Feature Engineering)
    .withColumn("age", F.floor(F.datediff(F.current_date(), F.col("birthday")) / 365.25))
    .withColumn("age_group", 
        F.when(F.col("age") < 30, "Under 30")
         .when(F.col("age") <= 50, "30 - 50")
         .otherwise("Over 50")
    )
    # 6. Cập nhật tem Audit
    .withColumn("_updated_at", F.current_timestamp())
    .drop("_ingested_at", "_source_file")
)

# Ghi ra bảng Delta tầng Silver
(df_silver_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.silver_customers"))

print(" ĐÃ TẠO THÀNH CÔNG BẢNG SẠCH: workspace.silver.silver_customers (15,266 khách hàng)!")

# Hiển thị 5 khách hàng đã có tuổi và nhóm tuổi
display(spark.table("workspace.silver.silver_customers").select("customer_key", "name", "birthday", "age", "age_group", "city", "country").limit(5))


In [0]:
import pyspark.sql.functions as F

df_sales_bronze = spark.table("workspace.bronze.bronze_sales")

# BỆNH 1: HỖN LOẠN TIỀN TỆ (CURRENCY CHAOS)
print("=== 1. ĐƠN HÀNG BÁN BẰNG 5 ĐỒNG TIỀN KHÁC NHAU ===")
df_sales_bronze.groupBy("currency_code").count().show()

# BỆNH 2: NGÀY GIAO HÀNG BỊ NULL VÀ FORMAT NGÀY MỸ (M/d/yyyy)
print("=== 2. SOI FORMAT NGÀY VÀ CÁC ĐƠN ĐANG GIAO (DELIVERY DATE BỊ NULL) ===")
display(df_sales_bronze.select("order_number", "order_date", "delivery_date", "product_key", "quantity", "currency_code").limit(5))


In [0]:
import pyspark.sql.functions as F

# 1. Đọc dữ liệu và chuẩn hóa tên khóa sang snake_case
df_sales_bronze = spark.table("workspace.bronze.bronze_sales")

# Đổi productkey -> product_key trong bảng Products
df_products = (spark.table("workspace.silver.silver_products")
    .withColumnRenamed("productkey", "product_key"))

# 2. XỬ LÝ LÀM SẠCH VÀ TÍNH TOÁN TÀI CHÍNH
df_silver_sales = (df_sales_bronze
    # Chuẩn hóa tên khóa ngoại trong Sales
    .withColumnRenamed("customerkey", "customer_key")
    .withColumnRenamed("storekey", "store_key")
    .withColumnRenamed("productkey", "product_key")
    
    # Khử trùng lặp theo cặp khóa tự nhiên (order_number + line_item)
    .dropDuplicates(["order_number", "line_item"])
    
    # Chuẩn hóa ngày tháng
    .withColumn("order_date", F.to_date(F.col("order_date"), "M/d/yyyy"))
    .withColumn("delivery_date", F.to_date(F.col("delivery_date"), "M/d/yyyy"))
    
    # Tính số ngày giao hàng và Trạng thái đơn hàng
    .withColumn("delivery_days", F.datediff(F.col("delivery_date"), F.col("order_date")))
    .withColumn("order_status", 
        F.when(F.col("delivery_date").isNull(), "In Transit")
         .otherwise("Delivered")
    )
    
    # BROADCAST JOIN với bảng Sản phẩm để lấy giá bán và giá vốn
    .join(
        F.broadcast(df_products.select("product_key", "unit_price_usd", "unit_cost_usd")),
        on="product_key", 
        how="inner"
    )
    
    # TÍNH TOÁN CÁC CHỈ SỐ DOANH THU & LỢI NHUẬN USD
    .withColumn("sales_amount_usd", F.round(F.col("quantity") * F.col("unit_price_usd"), 2))
    .withColumn("total_cost_usd", F.round(F.col("quantity") * F.col("unit_cost_usd"), 2))
    .withColumn("gross_profit_usd", F.round(F.col("sales_amount_usd") - F.col("total_cost_usd"), 2))
    
    # Cập nhật tem Audit
    .withColumn("_updated_at", F.current_timestamp())
    .drop("_ingested_at", "_source_file")
)

# 3. Ghi ra Delta Table tầng Silver
(df_silver_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.silver_sales"))

print(" ĐÃ TẠO THÀNH CÔNG BẢNG SẠCH: workspace.silver.silver_sales (62,884 đơn hàng)!")

# 4. Hiển thị 5 dòng đơn hàng đã có Doanh thu USD và Lợi nhuận USD
display(spark.table("workspace.silver.silver_sales").select(
    "order_number", "order_date", "order_status", "quantity", 
    "unit_price_usd", "sales_amount_usd", "gross_profit_usd"
).limit(5))



In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# 1. Đọc bảng Sales và Stores từ tầng Silver lên
df_sales = spark.table("workspace.silver.silver_sales")
df_stores = spark.table("workspace.silver.silver_stores")

# 2. Tính tổng doanh thu theo từng Cửa hàng (kèm thông tin Quốc gia)
df_store_revenue = (df_sales
    .join(df_stores, on="store_key", how="inner")
    .groupBy("country", "store_key", "store_size_category")
    .agg(F.round(F.sum("sales_amount_usd"), 2).alias("total_revenue_usd"))
)

# 3. ĐỊNH NGHĨA WINDOW SPEC: Phân vùng theo Quốc gia, xếp hạng theo Doanh thu giảm dần
window_spec = Window.partitionBy("country").orderBy(F.col("total_revenue_usd").desc())

# 4. ÁP DỤNG DENSE_RANK VÀ LỌC LẤY TOP 3
df_top3_stores = (df_store_revenue
    .withColumn("rank", F.dense_rank().over(window_spec))
    .filter(F.col("rank") <= 3)
    .orderBy("country", "rank")
)

print(" TOP 3 CỬA HÀNG DOANH THU CAO NHẤT TỪNG QUỐC GIA:")
display(df_top3_stores)


In [0]:
import pyspark.sql.functions as F

# 1. TẠO SCHEMA GOLD
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

# 2. BẢNG DIM 1: dim_store (Chính là bài toán D_SHOP_FLC)
(spark.table("workspace.silver.silver_stores")
    .select("store_key", "country", "state", "square_meters", "open_date", "store_size_category")
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_store"))
print(" Đã tạo bảng: workspace.gold.dim_store (D_SHOP)")

# 3. BẢNG DIM 2: dim_product (Đổi productkey -> product_key)
(spark.table("workspace.silver.silver_products")
    .withColumnRenamed("productkey", "product_key")
    .select("product_key", "product_name", "brand", "color", "unit_cost_usd", "unit_price_usd", "subcategory", "category")
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_product"))
print(" Đã tạo bảng: workspace.gold.dim_product")

# 4. BẢNG DIM 3: dim_customer
(spark.table("workspace.silver.silver_customers")
    .select("customer_key", "name", "gender", "city", "country", "birthday", "age", "age_group")
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_customer"))
print(" Đã tạo bảng: workspace.gold.dim_customer")

# 5. BẢNG DIM 4: dim_date (Tự động sinh ra lịch ngày để Power BI làm Time Intelligence)
df_dates = (spark.table("workspace.silver.silver_sales")
    .select(F.col("order_date").alias("date"))
    .distinct()
    .filter(F.col("date").isNotNull())
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.concat(F.lit("Q"), F.quarter("date")))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day_of_month", F.dayofmonth("date"))
    .withColumn("day_of_week", F.date_format("date", "EEEE"))
)
(df_dates.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_date"))
print(" Đã tạo bảng: workspace.gold.dim_date")

print("\n🎉 HOÀN TẤT 100% CẢ 4 BẢNG DIMENSION TẦNG GOLD!")



In [0]:
# TẠO BẢNG FACT BÁN HÀNG TRUNG TÂM (TẦNG GOLD)
df_fact = (spark.table("workspace.silver.silver_sales")
    .select(
        "order_number", 
        "line_item", 
        "order_date", 
        "delivery_date", 
        "store_key", 
        "customer_key", 
        "product_key", 
        "quantity", 
        "currency_code", 
        "unit_price_usd", 
        "unit_cost_usd", 
        "sales_amount_usd", 
        "total_cost_usd", 
        "gross_profit_usd", 
        "delivery_days", 
        "order_status"
    )
)

# Ghi ra bảng Delta tầng Gold kèm Liquid Clustering
(df_fact.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .clusterBy("store_key", "order_date")   # TỐI ƯU HÓA HIỆU NĂNG CHO POWER BI
    .saveAsTable("workspace.gold.fact_sales"))

print(" ĐÃ TẠO THÀNH CÔNG BẢNG FACT: workspace.gold.fact_sales (62,884 dòng)!")

# Xem thử 5 dòng Fact hoàn chỉnh
display(spark.table("workspace.gold.fact_sales").limit(5))
